# CommerceLens documentation evaluations

Run factual Q&A and high-level plan evaluations against the same assistant as the UI. Credentials remain in CommerceLens/.env. See docs/RAG_RUNBOOK.md. External judging and LangSmith upload are opt-in.

In [ ]:
from pathlib import Path
import subprocess, json
ROOT = Path.cwd().resolve()
if ROOT.name == "evals": ROOT = ROOT.parent
if ROOT.name != "CommerceLens": ROOT = ROOT / "CommerceLens"
PYTHON = ROOT / ".ai-venv/Scripts/python.exe"
assert PYTHON.exists(), "Install requirements-ai.txt into .ai-venv first"
def run(*args):
    result = subprocess.run([str(PYTHON), "-m", "commercelens_ai", *args], cwd=ROOT, text=True, encoding="utf-8", capture_output=True)
    print(result.stdout)
    if result.returncode: print(result.stderr); raise RuntimeError(f"CLI failed: {result.returncode}")
run("status")

## Build local artifacts
This does not contact Pinecone or an LLM. Ingest explicitly after setup when documentation changes.

In [ ]:
run("build")
# run("setup")
# run("ingest")

In [ ]:
run("ask", "Which service owns authoritative product prices and stock?")
run("plan", "Plan high-level support for cancellation before shipment")

## Repeated evaluation
Set the flags deliberately. Judge/RAGAS use the configured model provider; LangSmith uploads examples, evidence and results. Plans use rubric judging, not reference-based RAGAS scoring.

In [ ]:
REPETITIONS = 3
ENABLE_JUDGE = False
ENABLE_RAGAS = False
UPLOAD_LANGSMITH = False
OUTPUT = ROOT / "evals/results/notebook-run"
args = ["evaluate", "--repetitions", str(REPETITIONS), "--output", str(OUTPUT)]
if ENABLE_JUDGE: args.append("--judge")
if ENABLE_RAGAS: args.append("--ragas")
if UPLOAD_LANGSMITH: args.append("--langsmith")
run(*args)

In [ ]:
summary = json.loads((OUTPUT / "summary.json").read_text())
summary

In [ ]:
runs = [json.loads(line) for line in (OUTPUT / "runs.jsonl").read_text().splitlines()]
[{"case":r["case_id"], "repeat":r["repetition"], "status":r["status"], "judge":r["judge"]["status"], "ragas":r["ragas"]["status"]} for r in runs]